#Scraping town board meeting transcripts from youtube channel (mainly for practice): 
https://eastfishkillny.gov/agendas-minutes/
(* trainscriptions on Youtube channel)
Town Board	Jun 25, 2026 - 6:00 PM	Agenda	 	Agenda Packet
Town Board	May 21, 2026 - 6:00 PM	Agenda	 	Agenda Packet
Town Board	May  7, 2026 - 6:00 PM	Agenda	 	Agenda Packet
Town Board	Apr 23, 2026 - 6:00 PM	Agenda	 	Agenda Packet
(* transcriptions available online)
Town Board	Mar 26, 2026 - 6:00 PM	Agenda	 	Agenda Packet
Town Board	Feb 12, 2026 - 6:00 PM	
Town Board	Jan 15, 2026 - 6:00 PM	
https://www.youtube.com/@EF22NY

The town posted the minutes of 3 earlier board meetings online so i didn't need to scrape those: https://eastfishkillny.gov/wp-content/uploads/2026/03/01152026-Town-Board-Minutes.pdf https://eastfishkillny.gov/wp-content/uploads/2026/04/TB-February-12-2026-002-1.pdf https://eastfishkillny.gov/wp-content/uploads/2026/04/TB-March-26-2026.pdf

#Main steps: 
--Step 1: Collect Video IDs since March 2026
use the lightweight python command-line library yt-dlp. It fetches a clean text manifest file containing every video ID and its upload timestamp from the channel page.
--Step 2: Grab Raw Text Without Scraping
use the open-source youtube-transcript-api package. This library makes direct backend data network requests to YouTube's hidden caption servers, returning clean text loops instantly.
--Step 3: Write 

In [1]:
#sources: I actually got the wrong ulr for the March meeting somehow, so had to re-run everything to get the correct transcript. oh dear
#Jan-March meeting minutes online: https://eastfishkillny.gov/agendas-minutes/

#doubchecking: only need to scrape 4 meetings since March (*no meeting in April and two meetings in May): 
#Town Board Meeting Minutes March 26, 2026
#https://www.youtube.com/watch?v=1Z93ouGDtcE
#Town Board Meeting Minutes May 7, 2026
#https://www.youtube.com/watch?v=QdSmt54mGxQ
#Town Board Meeting Minutes May 21, 2026
#https://www.youtube.com/watch?v=g32Dn-tm6qQ
# Town Board Meeting Minutes June 25, 2026
#https://www.youtube.com/watch?v=TbMaNRm7Sq0

import pandas as pd
from youtube_transcript_api import YouTubeTranscriptApi

video_manifest = [
    {"date": "2026-03-26", "id": "1Z93ouGDtcE", "title": "Town Board Meeting"},
    {"date": "2026-05-07", "id": "QdSmt54mGxQ", "title": "Town Board Meeting"},
    {"date": "2026-05-21", "id": "g32Dn-tm6qQ", "title": "Town Board Meeting"},
    {"date": "2026-06-25", "id": "TbMaNRm7Sq0", "title": "Town Board Meeting"}
]

ytt_api = YouTubeTranscriptApi()  # create the API object once, reuse it below

all_transcript_data = []
for video in video_manifest:
    try:
        # .fetch() is the current method name (replaces the old .get_transcript())
        # .to_raw_data() converts it into the same simple list of dicts
        raw_transcript = ytt_api.fetch(video["id"]).to_raw_data()

        full_text = " ".join([segment["text"] for segment in raw_transcript])

        all_transcript_data.append({
            "Month": pd.to_datetime(video["date"]).strftime("%B"),
            "Meeting Title": video["title"],
            "Full Text": full_text.lower()
        })
        print(f"Successfully processed: {video['date']} ({video['id']})")
    except Exception as e:
        # Print the ACTUAL error instead of a fixed guess -- this is the
        # important fix. Without this, every failure looks identical
        # whether it's a version issue, no captions, or something else.
        print(f"Skipping video {video['id']}: {type(e).__name__}: {e}")

df = pd.DataFrame(all_transcript_data)
df



Successfully processed: 2026-03-26 (1Z93ouGDtcE)
Successfully processed: 2026-05-07 (QdSmt54mGxQ)
Successfully processed: 2026-05-21 (g32Dn-tm6qQ)
Successfully processed: 2026-06-25 (TbMaNRm7Sq0)


,Month,Meeting Title,Full Text
0,March,Town Board Meeting,good evening everyone. thanks for joining us. ...
1,May,Town Board Meeting,"good evening, everyone. thanks for joining us...."
2,May,Town Board Meeting,good evening everyone. thanks for joining us. ...
3,June,Town Board Meeting,"good evening. good evening, everyone. thanks f..."


In [ ]:
# Confirm what's actually in memory RIGHT NOW, before doing anything else
print(df.shape)          # should be (4, 3) -- 4 rows, 3 columns
print(df["Month"].tolist())   # should show March, May, May, June -- no April

(4, 3)
['March', 'May', 'May', 'June']


In [3]:
df.to_csv("town_board_transcripts.csv", index=False)
print("Saved to town_board_transcripts.csv")

Saved to town_board_transcripts.csv


In [4]:
df["char_count"] = df["Full Text"].str.len()
df[["Month", "char_count"]]

,Month,char_count
0,March,104212
1,May,49615
2,May,46405
3,June,143990


#My goal is to use this text data to somehow generate a SVG chart, to potentially show anexity level rising about the proposed data center (didn't end up doing that. went with a wordcloud thing instead)